# Text-to-Speech synthesis using Kokoro and OpenVINO GenAI

[Kokoro](https://huggingface.co/hexgrad/Kokoro-82M) is an open-weight text-to-speech model with 82 million parameters. Despite its lightweight architecture, it provides high-quality speech generation while remaining fast and resource-efficient.

[OpenVINO GenAI](https://github.com/openvinotoolkit/openvino.genai) provides an end-to-end `Text2SpeechPipeline` for Kokoro, including text-to-phoneme conversion, device-specific model compilation, and audio generation. This tutorial demonstrates how to use it with a ready-to-run [INT8 OpenVINO model](https://huggingface.co/OpenVINO/Kokoro-82M-int8-ov) or with a model exported locally.

<div class="alert alert-block alert-info"><b>Python support:</b> Inference with the preconverted model supports Python 3.10 and newer, including Python 3.13. Local export uses the <code>kokoro</code> and <code>misaki</code> Python packages and therefore requires Python &lt; 3.13.</div>

<div class="alert alert-block alert-warning"><b>NPU support:</b> Kokoro requires OpenVINO 2026.3 or later and a compatible NPU driver. See the <a href="https://docs.openvino.ai/2026/openvino-workflow/running-inference/inference-devices-and-modes/npu-device.html">NPU device guide</a>.</div>

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Select model source](#Select-model-source)
- [Download or convert the model](#Download-or-convert-the-model)
- [Prepare the OpenVINO GenAI pipeline](#Prepare-the-OpenVINO-GenAI-pipeline)
    - [Select inference device](#Select-inference-device)
- [Run inference](#Run-inference)
- [Interactive Demo](#Interactive-Demo)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

The notebook installs `espeakng-loader` and configures its bundled `espeak-ng` library. OpenVINO GenAI uses `espeak-ng` as a fallback for unknown English words and as the primary text-to-phoneme engine for supported non-English languages.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/kokoro/kokoro.ipynb" />

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from pathlib import Path
import requests

if not Path("pip_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    )
    open("pip_helper.py", "w").write(r.text)

from pip_helper import pip_install

pip_install(
    "-q",
    "openvino>=2026.3.0",
    "openvino-genai>=2026.3.0",
    "huggingface-hub",
    "soundfile",
    "gradio>=4.19",
    "ipywidgets",
    "espeakng-loader>=0.2.4",
)

if not Path("notebook_utils.py").exists():
    r = requests.get("https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    with open("notebook_utils.py", "w") as f:
        f.write(r.text)

## Select model source
[back to top ⬆️](#Table-of-contents:)

Select FP16, INT8, or INT4 weights. FP16 provides the highest fidelity, while INT8 and INT4 reduce model size and may improve performance at the cost of some accuracy.

A ready-to-run INT8 model is currently available as [OpenVINO/Kokoro-82M-int8-ov](https://huggingface.co/OpenVINO/Kokoro-82M-int8-ov), so INT8 is selected and downloaded by default. For every format, the notebook checks the corresponding OpenVINO repository and uses it when available. If it is unavailable or **Use preconverted model** is disabled, Optimum Intel exports the selected format locally.

Local export installs the original `kokoro` and `misaki` packages and is available only on Python 3.10–3.12.

In [ ]:
# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("kokoro.ipynb")

import sys
import ipywidgets as widgets
from IPython.display import display

weight_format = widgets.Dropdown(
    options=["FP16", "INT8", "INT4"],
    value="INT8",
    description="Weight format:",
)
use_preconverted = widgets.Checkbox(
    value=True,
    description="Use preconverted model when available",
)

if sys.version_info >= (3, 13):
    print("Python 3.13 or newer detected: formats without a preconverted model cannot be exported locally because export dependencies require Python < 3.13.")

display(weight_format, use_preconverted)

## Download or convert the model
[back to top ⬆️](#Table-of-contents:)

When the preconverted option is enabled and the selected format is available on Hugging Face Hub, `snapshot_download` retrieves the complete GenAI-compatible directory, including the OpenVINO IR, Kokoro dictionaries, and voice embeddings.

Otherwise, the notebook exports `hexgrad/Kokoro-82M` with the current Optimum Intel exporter and passes the selected `fp16`, `int8`, or `int4` weight format. The resulting directory has the same layout expected by `Text2SpeechPipeline`.

In [ ]:
import huggingface_hub as hf_hub

model_id = "hexgrad/Kokoro-82M"
selected_format = weight_format.value
ov_model_ids = {
    "FP16": "OpenVINO/Kokoro-82M-fp16-ov",
    "INT8": "OpenVINO/Kokoro-82M-int8-ov",
    "INT4": "OpenVINO/Kokoro-82M-int4-ov",
}
ov_model_id = ov_model_ids[selected_format]


def model_is_ready(path: Path) -> bool:
    required_files = [
        path / "openvino_model.xml",
        path / "openvino_model.bin",
        path / "config.json",
        path / "voices" / "af_heart.bin",
        path / "data" / "us_gold.json",
    ]
    return all(file.exists() for file in required_files)


use_hub_model = use_preconverted.value and hf_hub.repo_exists(ov_model_id)

if use_hub_model:
    model_dir = Path(ov_model_id.split("/")[-1])
    if not model_is_ready(model_dir):
        print(f"Downloading the preconverted {selected_format} model from {ov_model_id}...")
        hf_hub.snapshot_download(repo_id=ov_model_id, local_dir=model_dir)
else:
    if use_preconverted.value:
        print(f"A preconverted {selected_format} model is unavailable. Falling back to local export.")
    if sys.version_info >= (3, 13):
        raise RuntimeError(
            f"No preconverted {selected_format} model is available, and local Kokoro export requires Python < 3.13 "
            "because the kokoro and misaki packages do not support Python 3.13 yet. Select INT8 or use Python 3.10–3.12."
        )

    model_dir = Path("Kokoro-82M-ov") / selected_format
    if not model_is_ready(model_dir):
        pip_install(
            "-q",
            "-U",
            "git+https://github.com/huggingface/optimum-intel.git",
            "kokoro>=0.9.4",
            "misaki[en]>=0.9.4",
            "torch",
            "--extra-index-url",
            "https://download.pytorch.org/whl/cpu",
        )

        if not Path("cmd_helper.py").exists():
            r = requests.get("https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
            with open("cmd_helper.py", "w") as f:
                f.write(r.text)

        from cmd_helper import optimum_cli

        optimum_cli(
            model_id,
            model_dir,
            additional_args={"trust-remote-code": "", "weight-format": selected_format.lower()},
        )

print(f"Model directory: {model_dir.resolve()}")

## Prepare the OpenVINO GenAI pipeline
[back to top ⬆️](#Table-of-contents:)

`openvino_genai.Text2SpeechPipeline` loads the exported Kokoro model and provides end-to-end text-to-speech generation. A Kokoro voice pack stores one 256-value style vector for every supported phoneme-sequence length. The helper below validates the selected `.bin` file against the shape expected by the pipeline and creates an `ov.Tensor`.

The bundled `espeak-ng` library is configured before pipeline construction so it can provide fallback text-to-phoneme conversion.

In [ ]:
import os
import numpy as np
import openvino as ov
import openvino_genai as ov_genai
import espeakng_loader

os.environ.setdefault("MISAKI_ESPEAK_LIBRARY", espeakng_loader.get_library_path())
os.environ.setdefault("ESPEAK_DATA_PATH", espeakng_loader.get_data_path())


def load_voice_embedding(voice_path: Path, expected_shape: ov.Shape) -> ov.Tensor:
    data = np.fromfile(voice_path, dtype=np.float32)
    expected_size = int(np.prod(expected_shape))
    if data.size != expected_size:
        raise ValueError(f"Voice embedding has {data.size} values, but the model expects {expected_size} for shape {tuple(expected_shape)}")
    return ov.Tensor(data.reshape(expected_shape))


def get_audio_data(speech: ov.Tensor) -> np.ndarray:
    """Copy an output tensor to host memory when inference uses a remote device tensor."""
    try:
        return np.asarray(speech.data, dtype=np.float32).reshape(-1)
    except RuntimeError:
        host_tensor = ov.Tensor(speech.element_type, speech.shape)
        speech.copy_to(host_tensor)
        return np.asarray(host_tensor.data, dtype=np.float32).reshape(-1)

### Select inference device
[back to top ⬆️](#Table-of-contents:)

OpenVINO GenAI applies Kokoro-specific compilation settings automatically. This includes FP32 inference precision on GPU and the required static-shape configuration on NPU.

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="CPU")
device

## Run inference
[back to top ⬆️](#Table-of-contents:)

Create the GenAI pipeline and select one of the voice embeddings included with the model. This example uses American English (`en-us`); other currently supported language values are `en-gb`, `es`, `fr-fr`, `hi`, `it`, and `pt-br`. Choose a matching voice for the selected language.

In [ ]:
pipe = ov_genai.Text2SpeechPipeline(model_dir, device.value)

voice = widgets.Dropdown(
    options=[
        ("Heart", "af_heart"),
        ("Bella", "af_bella"),
        ("Nicole", "af_nicole"),
        ("Michael", "am_michael"),
        ("Fenrir", "am_fenrir"),
    ],
    value="af_heart",
    description="Voice:",
)
speed = widgets.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.1, description="Speed:")

display(voice, speed)

In [ ]:
from IPython.display import Audio, display

input_text = "Kokoro is an open-weight text-to-speech model with 82 million parameters. OpenVINO GenAI provides efficient speech generation on Intel hardware."
voice_path = model_dir / "voices" / f"{voice.value}.bin"
speaker_embedding = load_voice_embedding(voice_path, pipe.get_speaker_embedding_shape())

result = pipe.generate(
    input_text,
    speaker_embedding,
    language="en-us",
    speed=float(speed.value),
)

audio_data = get_audio_data(result.speeches[0])
if audio_data.size == 0 or not np.isfinite(audio_data).all():
    raise RuntimeError("The generated waveform is empty or contains non-finite values")

print(f"Generated {audio_data.size} samples at {result.output_sample_rate} Hz")
display(Audio(data=audio_data, rate=result.output_sample_rate))

## Interactive Demo
[back to top ⬆️](#Table-of-contents:)

In [ ]:
if not Path("gradio_helper.py").exists():
    r = requests.get("https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/kokoro/gradio_helper.py")
    with open("gradio_helper.py", "w") as f:
        f.write(r.text)

from gradio_helper import make_demo

demo = make_demo(pipe, model_dir)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(share=True, debug=True)
# If you are launching remotely, specify server_name and server_port.
# demo.launch(server_name="your server name", server_port=your_port)
# Read more in the docs: https://gradio.app/docs/

In [ ]:
# Cleanup (uncomment if these packages are no longer needed in the current environment)
# %pip uninstall -y -q openvino-genai openvino-tokenizers espeakng-loader kokoro misaki